# DEU Two-Clocks public verification notebook

This notebook verifies the summary-table arithmetic used in the draft manuscript. It is designed for author-led public verification, not as a substitute for independent peer review.

Scope reminders:

- The Bayes-factor values are conditional on the staged likelihood, prior, chain, and estimator design.
- `TT_lite` is diagnostic-only and excluded from the sampled production posterior/evidence.
- If raw chain files are absent, this notebook verifies the released summary tables. If compatible chain files are present, it also attempts an optional chain-level summary.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

print(f'ROOT = {ROOT}')
print(f'DATA = {DATA}')


In [ ]:
posterior = pd.read_csv(DATA / 'deu_posterior_summary.csv')
map_row = pd.read_csv(DATA / 'deu_map_like_row.csv')
native = pd.read_csv(DATA / 'native_comparison.csv')
evidence = pd.read_csv(DATA / 'evidence_lineage.csv')
stages = pd.read_csv(DATA / 'stage_readiness.csv')
legacy = pd.read_csv(DATA / 'legacy_background_screens.csv')

display(posterior)
display(native)
display(evidence)


## Posterior-to-lock offsets

The manuscript emphasizes that the production posterior means stayed close to the full-objective working lock. The table below computes those offsets directly from the summary CSV.


In [ ]:
posterior['mean_minus_lock'] = posterior['mean'] - posterior['working_lock']
posterior['fractional_offset'] = posterior['mean_minus_lock'] / posterior['working_lock']
posterior_offsets = posterior[['parameter', 'mean', 'working_lock', 'mean_minus_lock', 'fractional_offset']]
display(posterior_offsets)


## Evidence-lineage arithmetic

The log Bayes factor is recomputed as



`log_BF = DEU_log_evidence - native_log_evidence`.


In [ ]:
evidence_check = evidence.copy()
evidence_check['recomputed_log_bf'] = evidence_check['deu_log_evidence'] - evidence_check['native_log_evidence']
evidence_check['absolute_error'] = (evidence_check['recomputed_log_bf'] - evidence_check['log_bf_deu_minus_native']).abs()
display(evidence_check[['lineage', 'deu_log_evidence', 'native_log_evidence', 'log_bf_deu_minus_native', 'recomputed_log_bf', 'absolute_error']])

assert evidence_check['absolute_error'].max() < 1e-10, 'Evidence arithmetic mismatch.'
formal = evidence_check.loc[evidence_check['lineage'] == 'formal_proxy', 'log_bf_deu_minus_native'].iloc[0]
independent = evidence_check.loc[evidence_check['lineage'] == 'independent_validation', 'log_bf_deu_minus_native'].iloc[0]
print(f'formal/proxy log BF = {formal:.12f}')
print(f'independent-validation log BF = {independent:.12f}')
print(f'independent minus formal = {independent - formal:.12f}')


## Readiness and scope checks

These checks are intentionally simple. They confirm that the summary ledger records the stated author-led gates and the diagnostic-only `TT_lite` status.


In [ ]:
def stage_value(item: str):
    rows = stages.loc[stages['item'] == item, 'value']
    return None if rows.empty else rows.iloc[0]

required_true = [
    'posterior_readiness_gate_pass',
    'posterior_interpretation_allowed',
    'segment_gate_pass',
    'tail_gate_pass',
    'ess_gate_pass',
    'tri_anchor_monitoring_pass',
    'stage3b10_summary_reproduced',
    'split_robustness_gate_pass',
    'point_evaluations_success',
    'native_readiness_gate_pass',
    'native_config_has_no_deu_keys',
    'native_tt_lite_excluded',
]

for item in required_true:
    value = str(stage_value(item)).lower()
    print(f'{item}: {value}')
    assert value == 'true', f'{item} was not recorded as true.'

print('TT_lite diagnostic recorded:', stage_value('TT_lite_diagnostic_recorded'))
print('Native TT_lite excluded:', stage_value('native_tt_lite_excluded'))


## Optional chain-level summary

If compatible raw chain files are placed in `data/chains/deu/` and `data/chains/native/`, the helper below attempts to load them. This is deliberately conservative because Cobaya chain-column formats may vary. If the raw chains are absent, the notebook reports that it used summary-table verification only.


In [ ]:
def load_chain_directory(path: Path):
    files = sorted([p for p in path.glob('*') if p.suffix.lower() in {'.txt', '.csv', '.dat'} and not p.name.startswith('.')])
    if not files:
        return None, []
    frames = []
    used = []
    for fp in files:
        try:
            # Cobaya text chains often have whitespace-separated columns. CSV summaries use commas.
            if fp.suffix.lower() == '.csv':
                df = pd.read_csv(fp)
            else:
                df = pd.read_csv(fp, comment='#', delim_whitespace=True, engine='python')
            if len(df):
                frames.append(df)
                used.append(fp.name)
        except Exception as exc:
            print(f'Skipping {fp.name}: {exc}')
    if not frames:
        return None, used
    return pd.concat(frames, ignore_index=True), used

for label in ['deu', 'native']:
    chain, used = load_chain_directory(DATA / 'chains' / label)
    if chain is None:
        print(f'No compatible raw {label} chain files found. Using summary-table verification only.')
    else:
        print(f'Loaded {label} chain files: {used}')
        print(chain.head())
        common = [c for c in ['beta', 'p_vis', 'omega_b', 'n_s', 'A_planck'] if c in chain.columns]
        if common:
            display(chain[common].describe(percentiles=[0.16, 0.5, 0.84]).T)


## Write a compact verification report


In [ ]:
report = []
report.append('# DEU Two-Clocks verification report')
report.append('')
report.append('This report was generated by `notebooks/01_public_verification.ipynb`.')
report.append('')
report.append('## Evidence arithmetic')
for _, row in evidence_check.iterrows():
    report.append(f"- {row['lineage']}: log BF = {row['recomputed_log_bf']:.12f}")
report.append('')
report.append('## TT_lite scope')
report.append(f"- TT_lite diagnostic recorded: {stage_value('TT_lite_diagnostic_recorded')}")
report.append(f"- Native TT_lite excluded: {stage_value('native_tt_lite_excluded')}")
report.append('')
report.append('## Posterior means')
for _, row in posterior.iterrows():
    report.append(f"- {row['parameter']}: mean = {row['mean']}, working lock = {row['working_lock']}")

out = REPORTS / 'verification_report.md'
out.write_text('\n'.join(report) + '\n', encoding='utf-8')
print(f'Wrote {out}')
